In [ ]:
! python --version

In [8]:
from __future__ import annotations

"""Utility helpers for the recipe chatbot backend.

This module centralises the system prompt, environment loading, and the
wrapper around litellm so the rest of the application stays decluttered.
"""

import os
from typing import Final, List, Dict

import litellm  # type: ignore
from dotenv import load_dotenv

# Ensure the .env file is loaded as early as possible.
load_dotenv(override=False)

# --- Constants -------------------------------------------------------------------

meal_type_options = ['entrée', 'dessert', 'main', 'beverage']
dietary_preference_options = ['vegan', 'vegetarian', 'omnivore', 'meat', 'pescatarian', 'nut-allergic']
difficulty_options = ['easy', 'medium', 'hard', 'very_hard']
mealtime_options = ["breakfast", "lunch", "dinner", "snack"]
time_required_options = ['under_30_minutes', '30_to_60_minutes', 'over_1_hour']

SYSTEM_PROMPT: Final[str] = f'''\
Given the following key Japanese recipe dimensions:
- Difficulty: one of {difficulty_options}
- Dietary Preference: one of {dietary_preference_options}
- Meal Type: one of {meal_type_options}
- Mealtime: one of {mealtime_options}
- Time Required: one of {time_required_options}

Can you provide a list of exactly 20 combinations of these dimensions? They have to make sense together.

For each combination, also provide a brief persona description that would fit that combination.

For example, here are a few combinations based on a particular persona:

1) Persona: "Beginner cook looking for quick and easy meals"
   - Difficulty: easy
   - Dietary Preference: omnivore
   - Meal Type: main
   - Mealtime: dinner
   - Time Required: under_30_minutes

2) Persona: "Health-conscious individual seeking vegetarian options"
   - Difficulty: medium
   - Dietary Preference: vegetarian
   - Meal Type: entrée
   - Mealtime: lunch
   - Time Required: 30_to_60_minutes

3) Persona: "Gourmet chef interested in complex recipes"
    - Difficulty: very_hard
    - Dietary Preference: omnivore
    - Meal Type: main
    - Mealtime: dinner
    - Time Required: over_1_hour

Remember, that we only need 20 combinations.
'''

# Fetch configuration *after* we loaded the .env file.
MODEL_NAME: Final[str] = os.environ.get("MODEL_NAME", "gpt-4o-mini")

def get_dimension_combinations() -> list[tuple[str, tuple[str, str, str, str, str]]]:
    """
    Use the SYSTEM_PROMPT and an LLM call to return a realistic list of 20 unique combinations of personas, and key Japanese
    recipe dimensions (difficulty, dietary_preference, meal_type, mealtime, time_required) as tuples.
    
    Returns:
        List of tuples where each tuple is (persona_description, (difficulty, dietary_preference, meal_type, mealtime, time_required))
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Please provide the list as a Python list of tuples, where each tuple contains:\n(persona_description, (difficulty, dietary_preference, meal_type, mealtime, time_required))\n\nFor example:\n[('Beginner cook looking for quick meals', ('easy', 'omnivore', 'main', 'dinner', 'under_30_minutes')), ...]"}
    ]
    completion = litellm.completion(
        model=MODEL_NAME,
        messages=messages,
    )
    import ast
    import re
    # Extract the list of tuples from the assistant's reply
    reply = completion["choices"][0]["message"]["content"].strip()
    # Try to extract the first Python list of tuples from the reply
    match = re.search(r'\[.*\]', reply, re.DOTALL)
    if match:
        list_str = match.group(0)
        try:
            result = ast.literal_eval(list_str)
            if isinstance(result, list) and all(isinstance(t, tuple) and len(t) == 2 for t in result):
                # Validate structure: (persona_string, (5-tuple of dimensions))
                valid = all(
                    isinstance(t[0], str) and 
                    isinstance(t[1], tuple) and 
                    len(t[1]) == 5 
                    for t in result
                )
                if valid:
                    return result
        except Exception as e:
            print(f"Error parsing result: {e}")
            pass
    # Fallback: return the raw reply if parsing fails
    return reply


In [9]:
dimension_combinations = get_dimension_combinations()
length = len(dimension_combinations) if isinstance(dimension_combinations, list) else 'N/A'
print(f"Number of combinations: {length}")

Number of combinations: 20


In [10]:
dimension_combinations

[('Beginner cook looking for quick and easy meals',
  ('easy', 'omnivore', 'main', 'dinner', 'under_30_minutes')),
 ('Health-conscious vegetarian seeking nutritious lunch options',
  ('medium', 'vegetarian', 'entrée', 'lunch', '30_to_60_minutes')),
 ('Experienced cook wanting intricate seafood mains for dinner',
  ('very_hard', 'pescatarian', 'main', 'dinner', 'over_1_hour')),
 ('Vegan snack enthusiast who prefers fast recipes',
  ('easy', 'vegan', 'snack', 'snack', 'under_30_minutes')),
 ('Meat lover looking for traditional hearty breakfasts',
  ('medium', 'meat', 'main', 'breakfast', '30_to_60_minutes')),
 ('Nut-allergic individual planning elaborate desserts',
  ('hard', 'nut-allergic', 'dessert', 'dinner', 'over_1_hour')),
 ('Busy professional wanting easy vegan beverages',
  ('easy', 'vegan', 'beverage', 'breakfast', 'under_30_minutes')),
 ('Casual cook seeking simple pescatarian lunch mains',
  ('easy', 'pescatarian', 'main', 'lunch', 'under_30_minutes')),
 ('Vegetarian foodie ea

In [11]:
import asyncio

async def generate_single_query_async(persona: str, dimensions: tuple[str, str, str, str, str]) -> str:
    """Generate a single natural language query for one persona-dimension combination asynchronously."""
    difficulty, dietary_preference, meal_type, mealtime, time_required = dimensions
    
    prompt = f'''You are helping generate a realistic user query for a recipe chatbot.

Given this user persona: "{persona}"
And these recipe requirements:
- Difficulty level: {difficulty}
- Dietary preference: {dietary_preference}
- Meal type: {meal_type}
- Mealtime: {mealtime}
- Time required: {time_required}

Write a single, natural user query that someone with this persona might ask a recipe chatbot. The query should reflect their cooking level, dietary needs, and time constraints,
but should sound natural and conversational (not mentioning the specific dimension names). Natural language have a lot of imperfections, so feel free to include small typos or
colloquial phrases. Mis-spellings and informal language are welcome. Mixed capitalisations are also common in natural language queries. You don't have to allways start the
question with `Hey`. Also, some queries are quite terse and to the point, while others may be more elaborate.

Return only the query text, no additional formatting or explanation.'''
    
    messages = [
        {"role": "system", "content": "You are an expert at generating realistic, conversational user queries for a recipe chatbot."},
        {"role": "user", "content": prompt}
    ]
    
    try:
        completion = await asyncio.get_event_loop().run_in_executor(
            None, 
            lambda: litellm.completion(model=MODEL_NAME, messages=messages)
        )
        return completion["choices"][0]["message"]["content"].strip()
    except Exception as e:
        print(f"Error generating query for {persona[:30]}...: {e}")
        return f"Error generating query for {persona}"

async def generate_natural_language_queries_async(combinations: list[tuple[str, tuple[str, str, str, str, str]]], n: int = 10) -> list[str]:
    """Use async to generate realistic natural language user queries for each combination in parallel."""
    selected_combinations = combinations[:n]
    
    print(f"Generating {len(selected_combinations)} queries asynchronously...")
    
    tasks = [
        generate_single_query_async(persona, dimensions) 
        for persona, dimensions in selected_combinations
    ]
    
    queries = await asyncio.gather(*tasks)
    return queries

In [12]:
# In Jupyter, you can await async functions directly in cells
queries = await generate_natural_language_queries_async(dimension_combinations, n=20)
print(f"\nGenerated {len(queries)} user queries:")
print("="*50)
for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")
    print()

Generating 20 queries asynchronously...

Generated 20 user queries:
1. I’m new to cooking and need a simple chicken dinner recipe that I can whip up in like 20 mins tops. Any easy ideas?

2. I’m looking for a decent veggie lunch recipe that’s a bit challenging but won’t take longer than an hour to make. Got any good ideas?

3. I’m looking for a super detailed pescatarian seafood dish for dinner—something that’ll take me a good hour or more to make. Got any challenging recipes?

4. Got any quick and super easy vegan snack ideas? Need something I can whip up in like 20 mins or less.

5. I'm craving a solid meat-heavy breakfast, nothing too simple but not crazy hard either. Got like 30-60 mins maybe? Something traditional and filling would be perfect. Any ideas?

6. I’m looking to wow my dinner guests with a fancy dessert that takes a while to make but no nuts—something challenging enough to really show off my skills. Any ideas?

7. Got any quick and easy vegan smoothies or breakfast drin

In [13]:
# Export queries to CSV file
import csv
import os

# Create the data directory if it doesn't exist
os.makedirs('../../data', exist_ok=True)

# Write queries to CSV with the specified schema
csv_path = '../../data/sample_queries_hw2.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header
    writer.writerow(['id', 'query'])
    
    # Write queries with sequential IDs
    for i, query in enumerate(queries, 1):
        writer.writerow([i, query])

print(f"Exported {len(queries)} queries to {csv_path}")

Exported 20 queries to ../../data/sample_queries_hw2.csv


Now that the sample_queries_hw2.csv has been createed, run the bulk test script to generate results for homework 2:
```bash
python scripts/bulk_test_hw2.py
```

This will generate a new CSV file with the results of the bulk test for homework 2 in the `results/` directory. We can then run the data viewer annotation tool to review and annotate the results, for the Open Coding part of the analysis.
